# House AI — T4 image-to-3D backend

This version uses Stability AI SPAR3D instead of HunyuanWorld. SPAR3D supports a low-VRAM mode documented for GPUs with less than 7 GB VRAM, so your 14.6 GB Tesla T4 has substantially more memory than that mode targets. It reconstructs a textured 3D asset from the uploaded image; it does **not** invent a complete unseen house interior yet.


In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "GPU runtime required"
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory/1024**3, 1))


In [ ]:
!git clone --depth 1 https://github.com/Stability-AI/stable-point-aware-3d.git /content/stable-point-aware-3d
%cd /content/stable-point-aware-3d
!pip install -q -r requirements.txt
!pip install -q .


In [ ]:
!rm -rf /content/house-ai
!git clone --depth 1 https://github.com/Rohit9605/RohitGundam-house-ai.git /content/house-ai
import os, subprocess, time
os.environ["SPAR3D_DIR"]="/content/stable-point-aware-3d"
server=subprocess.Popen(["python","/content/house-ai/local-world/server.py"],env=os.environ.copy())
time.sleep(3)
print("House AI generation server started")


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
import subprocess, re, time
tunnel=subprocess.Popen(["/content/cloudflared","tunnel","--url","http://127.0.0.1:8787","--no-autoupdate"],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
url=None
deadline=time.time()+60
while time.time()<deadline:
    line=tunnel.stdout.readline()
    if line: print(line,end="")
    m=re.search(r"https://[a-z0-9-]+\\.trycloudflare\\.com",line)
    if m: url=m.group(0); break
assert url, "Cloudflare tunnel did not start"
print("\\nCOPY THIS URL INTO HOUSE AI:\\n"+url)
print("Keep this Colab runtime connected while generating.")
